# Práctica Completa: DataFrames en Apache Spark

Este documento presenta el desarrollo íntegro de la guía práctica de **DataFrames en Spark**, incluyendo:
- Los **4 Checkpoints** de validación teórica y práctica.
- Los **8 Ejercicios Prácticos** sobre creación, filtrado con operadores bitwise, transformación de columnas, agregaciones, funciones de ventana, cruces de tablas, consultas SQL e manipulación de cadenas.
- El **Proyecto Final: Análisis de Ventas** empleando el conjunto de datos real `pdi_sales.csv` con 841147 registros.

Toda la redacción explicativa se ha elaborado sin emplear paréntesis para hacer énfasis.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, count, avg, max, min, sum, desc,
    rank, broadcast, year, month, upper, split,
    when, trim, concat_ws, to_date
)
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Practica-DataFrames-Spark") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Sesión de Spark iniciada correctamente. Versión:", spark.version)

Sesión de Spark iniciada correctamente. Versión: 3.5.3


---
## Parte I: Checkpoints de la Guía

### Checkpoint 1: Creación, Selección y Filtrado
1. Crear un DataFrame con datos de empleados: `[("Ana", 25000, "Venta"), ("Luis", 32000, "IT")]`.
2. Seleccionar el nombre y crear una nueva columna `salario_futuro` aumentado un 10%.
3. Filtrar solo los empleados de `Venta` que ganen menos de 30000.

In [2]:
datos_cp1 = [("Ana", 25000, "Venta"), ("Luis", 32000, "IT")]
columnas_cp1 = ["nombre", "salario", "departamento"]
df_cp1 = spark.createDataFrame(datos_cp1, columnas_cp1)

print("1. DataFrame de empleados:")
df_cp1.show()

print("2. Selección de nombre y columna salario_futuro con incremento del 10%:")
df_cp1_futuro = df_cp1.select(
    col("nombre"),
    (col("salario") * 1.10).alias("salario_futuro")
)
df_cp1_futuro.show()

print("3. Filtro de empleados del departamento Venta que ganen menos de 30000:")
df_cp1_filtrado = df_cp1.filter((col("departamento") == "Venta") & (col("salario") < 30000))
df_cp1_filtrado.show()

1. DataFrame de empleados:
+------+-------+------------+
|nombre|salario|departamento|
+------+-------+------------+
|   Ana|  25000|       Venta|
|  Luis|  32000|          IT|
+------+-------+------------+

2. Selección de nombre y columna salario_futuro con incremento del 10%:
+------+--------------+
|nombre|salario_futuro|
+------+--------------+
|   Ana|       27500.0|
|  Luis|       35200.0|
+------+--------------+

3. Filtro de empleados del departamento Venta que ganen menos de 30000:
+------+-------+------------+
|nombre|salario|departamento|
+------+-------+------------+
|   Ana|  25000|       Venta|
+------+-------+------------+


### Checkpoint 2: Agregaciones
Agrupar por departamento y calcular tanto el salario medio como el número total de empleados por departamento.

In [3]:
df_cp2 = df_cp1.groupBy("departamento").agg(
    avg("salario").alias("salario_medio"),
    count("*").alias("total_empleados")
)
df_cp2.show()

+------------+-------------+---------------+
|departamento|salario_medio|total_empleados|
+------------+-------------+---------------+
|       Venta|      25000.0|              1|
|          IT|      32000.0|              1|
+------------+-------------+---------------+


### Checkpoint 3: Window Functions
Definir una ventana particionada por departamento y ordenada por salario de mayor a menor para clasificar a los empleados dentro de su respectivo departamento.

In [4]:
datos_cp3 = [
    ("Maria", "IT", 5000),
    ("Juan", "IT", 3000),
    ("Ana", "Ventas", 4500),
    ("Pedro", "Ventas", 4000)
]
df_cp3 = spark.createDataFrame(datos_cp3, ["nombre", "departamento", "salario"])
ventana_cp3 = Window.partitionBy("departamento").orderBy(col("salario").desc())
df_cp3_rank = df_cp3.withColumn("ranking_salario", rank().over(ventana_cp3))
df_cp3_rank.show()

+------+------------+-------+---------------+
|nombre|departamento|salario|ranking_salario|
+------+------------+-------+---------------+
| Maria|          IT|   5000|              1|
|  Juan|          IT|   3000|              2|
|   Ana|      Ventas|   4500|              1|
| Pedro|      Ventas|   4000|              2|
+------+------------+-------+---------------+


### Checkpoint 4: Joins
Para visualizar a todos los empleados tengan o no proyecto asignado, se utiliza un **Left Join** tomando como tabla izquierda el DataFrame de empleados.

In [5]:
df_emp_cp4 = spark.createDataFrame([(1, "Ana"), (2, "Luis"), (3, "Marta")], ["id", "nombre"])
df_proy_cp4 = spark.createDataFrame([(1, "Migracion Cloud"), (2, "IA Generativa")], ["id_empleado", "proyecto"])

df_join_cp4 = df_emp_cp4.join(
    df_proy_cp4,
    df_emp_cp4["id"] == df_proy_cp4["id_empleado"],
    "left"
).select(df_emp_cp4["id"], df_emp_cp4["nombre"], df_proy_cp4["proyecto"])
df_join_cp4.show()

+---+------+---------------+
| id|nombre|       proyecto|
+---+------+---------------+
|  1|   Ana|Migracion Cloud|
|  2|  Luis|  IA Generativa|
|  3| Marta|           NULL|
+---+------+---------------+


---
## Parte II: Ejercicios Prácticos

### Ejercicio 1: Creación y Exploración
Dada la lista de empleados:
`[("Ana", 28, "Ventas", 35000), ("Juan", 35, "IT", 45000), ("María", 42, "IT", 55000), ("Pedro", 31, "Ventas", 38000), ("Laura", 29, "Marketing", 40000)]`
1. Crear un DataFrame con columnas `nombre`, `edad`, `departamento`, `salario`.
2. Mostrar su esquema y las primeras 3 filas.
3. Contar el total de empleados.

In [6]:
datos_ej1 = [
    ("Ana", 28, "Ventas", 35000),
    ("Juan", 35, "IT", 45000),
    ("María", 42, "IT", 55000),
    ("Pedro", 31, "Ventas", 38000),
    ("Laura", 29, "Marketing", 40000)
]
columnas_ej1 = ["nombre", "edad", "departamento", "salario"]
df_empleados = spark.createDataFrame(datos_ej1, columnas_ej1)

print("Esquema del DataFrame:")
df_empleados.printSchema()

print("Primeras 3 filas:")
df_empleados.show(3)

print(f"Total de empleados: {df_empleados.count()}")

Esquema del DataFrame:
root
 |-- nombre: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- departamento: string (nullable = true)
 |-- salario: long (nullable = true)

Primeras 3 filas:
+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|   Ana|  28|      Ventas|  35000|
|  Juan|  35|          IT|  45000|
| María|  42|          IT|  55000|
+------+----+------------+-------+
only showing top 3 rows

Total de empleados: 5


### Ejercicio 2: Filtrado
Uso de operadores lógicos bitwise `&`, `|` y `~` para obtener:
1. Empleados de `IT` con salario mayor a 40000.
2. Empleados de `Ventas` o `Marketing`.
3. Empleados que no sean de `IT`.
4. Empleados entre 30 y 40 años.

In [7]:
print("1. Empleados de IT con salario > 40000:")
df_empleados.filter((col("departamento") == "IT") & (col("salario") > 40000)).show()

print("2. Empleados de Ventas o Marketing:")
df_empleados.filter((col("departamento") == "Ventas") | (col("departamento") == "Marketing")).show()

print("3. Empleados que NO son de IT:")
df_empleados.filter(~(col("departamento") == "IT")).show()

print("4. Empleados entre 30 y 40 años:")
df_empleados.filter((col("edad") >= 30) & (col("edad") <= 40)).show()

1. Empleados de IT con salario > 40000:
+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|  Juan|  35|          IT|  45000|
| María|  42|          IT|  55000|
+------+----+------------+-------+

2. Empleados de Ventas o Marketing:
+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|   Ana|  28|      Ventas|  35000|
| Pedro|  31|      Ventas|  38000|
| Laura|  29|   Marketing|  40000|
+------+----+------------+-------+

3. Empleados que NO son de IT:
+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|   Ana|  28|      Ventas|  35000|
| Pedro|  31|      Ventas|  38000|
| Laura|  29|   Marketing|  40000|
+------+----+------------+-------+

4. Empleados entre 30 y 40 años:
+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|  Juan|  35|          IT|  45000|
| Pedro|  31|    

### Ejercicio 3: Manipulación de Columnas
Añadir la columna `salario_en_5_años` calculada como `salario + salario * 0.05 * 5` y renombrar la columna `nombre` a `empleado`.

In [8]:
df_ej3 = df_empleados \
    .withColumn("salario_en_5_años", col("salario") + (col("salario") * 0.05 * 5)) \
    .withColumnRenamed("nombre", "empleado")
df_ej3.show()

+--------+----+------------+-------+-----------------+
|empleado|edad|departamento|salario|salario_en_5_años|
+--------+----+------------+-------+-----------------+
|     Ana|  28|      Ventas|  35000|          43750.0|
|    Juan|  35|          IT|  45000|          56250.0|
|   María|  42|          IT|  55000|          68750.0|
|   Pedro|  31|      Ventas|  38000|          47500.0|
|   Laura|  29|   Marketing|  40000|          50000.0|
+--------+----+------------+-------+-----------------+


### Ejercicio 4: Agregaciones
1. Calcular el salario promedio por departamento.
2. Obtener el salario máximo y mínimo por departamento.
3. Contar empleados por departamento.
4. Calcular el gasto total en salarios por departamento y ordenarlo de mayor a menor.

In [9]:
print("1. Salario promedio por departamento:")
df_empleados.groupBy("departamento").agg(avg("salario").alias("salario_promedio")).show()

print("2. Salario máximo y mínimo por departamento:")
df_empleados.groupBy("departamento").agg(
    max("salario").alias("salario_maximo"),
    min("salario").alias("salario_minimo")
).show()

print("3. Conteo de empleados por departamento:")
df_empleados.groupBy("departamento").count().show()

print("4. Gasto total en salarios por departamento ordenado de mayor a menor:")
df_empleados.groupBy("departamento") \
    .agg(sum("salario").alias("gasto_total_salarios")) \
    .orderBy(desc("gasto_total_salarios")) \
    .show()

1. Salario promedio por departamento:
+------------+----------------+
|departamento|salario_promedio|
+------------+----------------+
|      Ventas|         36500.0|
|          IT|         50000.0|
|   Marketing|         40000.0|
+------------+----------------+

2. Salario máximo y mínimo por departamento:
+------------+--------------+--------------+
|departamento|salario_maximo|salario_minimo|
+------------+--------------+--------------+
|      Ventas|         38000|         35000|
|          IT|         55000|         45000|
|   Marketing|         40000|         40000|
+------------+--------------+--------------+

3. Conteo de empleados por departamento:
+------------+-----+
|departamento|count|
+------------+-----+
|      Ventas|    2|
|          IT|    2|
|   Marketing|    1|
+------------+-----+

4. Gasto total en salarios por departamento ordenado de mayor a menor:
+------------+--------------------+
|departamento|gasto_total_salarios|
+------------+--------------------+
|       

### Ejercicio 5: Window Functions
Dadas las ventas mensuales `[("Enero", "ProdA", 100), ("Enero", "ProdB", 150), ("Febrero", "ProdA", 120)]`, crear un ranking de productos por ventas para cada mes.

In [10]:
datos_ventas_mes = [
    ("Enero", "ProdA", 100),
    ("Enero", "ProdB", 150),
    ("Febrero", "ProdA", 120)
]
df_ventas_mes = spark.createDataFrame(datos_ventas_mes, ["mes", "producto", "ventas"])
ventana_mes = Window.partitionBy("mes").orderBy(col("ventas").desc())
df_ranking_mes = df_ventas_mes.withColumn("ranking", rank().over(ventana_mes))
df_ranking_mes.show()

+-------+--------+------+-------+
|    mes|producto|ventas|ranking|
+-------+--------+------+-------+
|  Enero|   ProdB|   150|      1|
|  Enero|   ProdA|   100|      2|
|Febrero|   ProdA|   120|      1|
+-------+--------+------+-------+


### Ejercicio 6: Joins
Unir el DataFrame de personas con el DataFrame de departamentos mediante un **Left Join** y analizar qué ocurre con Marta.

In [11]:
df_personas = spark.createDataFrame(
    [("Ana", 1), ("Luis", 2), ("Marta", 3)],
    ["Nombre", "DepID"]
)
df_departamentos = spark.createDataFrame(
    [(1, "IT"), (2, "HR")],
    ["DepID", "NombreDep"]
)
df_join_ej6 = df_personas.join(df_departamentos, "DepID", "left")
df_join_ej6.show()

+-----+------+---------+
|DepID|Nombre|NombreDep|
+-----+------+---------+
|    1|   Ana|       IT|
|    2|  Luis|       HR|
|    3| Marta|     NULL|
+-----+------+---------+


**Respuesta sobre Marta:** Como se realiza un **Left Join**, todas las filas del DataFrame izquierdo que corresponde a Personas se conservan en el resultado. Dado que el `DepID` 3 de Marta no tiene coincidencia en el DataFrame de Departamentos, su registro aparece con el valor `NULL` en la columna `NombreDep`.

### Ejercicio 7: SQL Puro
Registrar el DataFrame del Ejercicio 1 como vista temporal y repetir el filtro de empleados de `IT` con salario mayor a 40000 empleando `spark.sql()`.

In [12]:
df_empleados.createOrReplaceTempView("empleados")
df_sql_ej7 = spark.sql("""
    SELECT nombre, edad, departamento, salario
    FROM empleados
    WHERE departamento = 'IT' AND salario > 40000
""")
df_sql_ej7.show()

+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|  Juan|  35|          IT|  45000|
| María|  42|          IT|  55000|
+------+----+------------+-------+


### Ejercicio 8: Funciones de String
Limpiar los espacios laterales con `trim` en ambos campos de `[(" Juan ", "García"), ("Ana", " Pérez ")]` y crear la columna `Nombre_Completo` concatenando nombre y apellido con un espacio intermedio.

In [13]:
datos_strings = [(" Juan ", "García"), ("Ana", " Pérez ")]
df_strings = spark.createDataFrame(datos_strings, ["nombre", "apellido"])
df_strings_limpio = df_strings \
    .withColumn("nombre", trim(col("nombre"))) \
    .withColumn("apellido", trim(col("apellido"))) \
    .withColumn("Nombre_Completo", concat_ws(" ", col("nombre"), col("apellido")))
df_strings_limpio.show()

+------+--------+---------------+
|nombre|apellido|Nombre_Completo|
+------+--------+---------------+
|  Juan|  García|    Juan García|
|   Ana|   Pérez|      Ana Pérez|
+------+--------+---------------+


---
## Parte III: Proyecto Final — Análisis de Ventas

Análisis sobre el archivo `DATOS/pdi_sales.csv`, el cual contiene los campos `ProductID`, `Date`, `Zip`, `Units`, `Revenue` y `Country` separados por punto y coma `;`.

### Paso 1 y Paso 2: Carga del archivo CSV, esquema y conteo total de filas

In [14]:
df_ventas = spark.read.csv(
    "DATOS/pdi_sales.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

df_ventas.printSchema()
print(f"Total de filas en pdi_sales.csv: {df_ventas.count()}")
df_ventas.show(5, truncate=False)

root
 |-- ProductID: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- Zip: string (nullable = true)
 |-- Units: integer (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- Country: string (nullable = true)

Total de filas en pdi_sales.csv: 841147
+---------+---------+---------------+-----+-------+-------+
|ProductID|Date     |Zip            |Units|Revenue|Country|
+---------+---------+---------------+-----+-------+-------+
|725      |1/15/1999|41540          |1    |115.5  |Germany|
|787      |6/6/2002 |41540          |1    |314.9  |Germany|
|788      |6/6/2002 |41540          |1    |314.9  |Germany|
|940      |1/15/1999|22587          |1    |687.7  |Germany|
|396      |1/15/1999|22587          |1    |857.1  |Germany|
+---------+---------+---------------+-----+-------+-------+
only showing top 5 rows


### Paso 3: Total de ingresos por país ordenado de mayor a menor
Se aplica `trim` sobre la columna `Country` para normalizar espacios en blanco al final de algunos nombres de país y se calcula la suma total de `Revenue`.

In [15]:
df_ingresos_pais = df_ventas.withColumn("Country", trim(col("Country"))) \
    .groupBy("Country") \
    .agg(sum("Revenue").alias("total_ingresos")) \
    .orderBy(desc("total_ingresos"))
df_ingresos_pais.show()

+-------+--------------------+
|Country|      total_ingresos|
+-------+--------------------+
| France|1.4849019329999772E8|
|Germany| 1.319305100999913E8|
| Mexico|1.0364580259999707E8|
| Canada| 3.411493090000375E7|
+-------+--------------------+


### Paso 4: Filtrar las ventas de Alemania con más de 1 unidad vendida

In [16]:
df_alemania_mas_1 = df_ventas.filter(
    (trim(col("Country")) == "Germany") & (col("Units") > 1)
)
print(f"Total de ventas en Alemania con más de 1 unidad: {df_alemania_mas_1.count()}")
df_alemania_mas_1.show(10, truncate=False)

Total de ventas en Alemania con más de 1 unidad: 7426
+---------+---------+---------------+-----+-------+-------+
|ProductID|Date     |Zip            |Units|Revenue|Country|
+---------+---------+---------------+-----+-------+-------+
|901      |2/15/1999|13587          |2    |818.9  |Germany|
|676      |1/15/1999|21107          |2    |1603.9 |Germany|
|604      |4/10/2003|10243          |2    |989.6  |Germany|
|491      |3/15/1999|10319          |2    |1630.7 |Germany|
|910      |3/15/1999|10435          |2    |829.4  |Germany|
|645      |3/15/1999|10435          |2    |1493.1 |Germany|
|862      |4/10/2003|10435          |2    |608.9  |Germany|
|499      |2/15/1999|10435          |2    |1692.6 |Germany|
|631      |2/15/1999|40880          |2    |692.5  |Germany|
|635      |6/3/2002 |12627          |2    |1716.2 |Germany|
+---------+---------+---------------+-----+-------+-------+
only showing top 10 rows


### Paso 5: Código postal con mayor número de ventas en Alemania
El código postal con mayor número de registros de venta en Alemania es `50169` con 2130 ventas.

In [17]:
df_top_zip_alemania = df_ventas.filter(trim(col("Country")) == "Germany") \
    .withColumn("Zip_limpio", trim(col("Zip"))) \
    .groupBy("Zip_limpio") \
    .agg(count("*").alias("total_ventas")) \
    .orderBy(desc("total_ventas"))
df_top_zip_alemania.show(5, truncate=False)

+----------+------------+
|Zip_limpio|total_ventas|
+----------+------------+
|50169     |2130        |
|52538     |2007        |
|40597     |1980        |
|20251     |1828        |
|22111     |1780        |
+----------+------------+
only showing top 5 rows


### Paso 6: Clasificación de cada venta según Revenue
Se añade la columna `categoria_venta` asignando el valor `Alta` cuando `Revenue > 500` y `Normal` en caso contrario.

In [18]:
df_clasificado = df_ventas.withColumn(
    "categoria_venta",
    when(col("Revenue") > 500, "Alta").otherwise("Normal")
)
df_clasificado.select("ProductID", "Date", "Units", "Revenue", "Country", "categoria_venta").show(10, truncate=False)
df_clasificado.groupBy("categoria_venta").count().show()

+---------+---------+-----+-------+-------+---------------+
|ProductID|Date     |Units|Revenue|Country|categoria_venta|
+---------+---------+-----+-------+-------+---------------+
|725      |1/15/1999|1    |115.5  |Germany|Normal         |
|787      |6/6/2002 |1    |314.9  |Germany|Normal         |
|788      |6/6/2002 |1    |314.9  |Germany|Normal         |
|940      |1/15/1999|1    |687.7  |Germany|Alta           |
|396      |1/15/1999|1    |857.1  |Germany|Alta           |
|734      |4/10/2003|1    |330.7  |Germany|Normal         |
|769      |2/15/1999|1    |257.2  |Germany|Normal         |
|499      |1/15/1999|1    |846.3  |Germany|Alta           |
|2254     |1/15/1999|1    |57.7   |Germany|Normal         |
|31       |5/31/2002|1    |761.2  |Germany|Alta           |
+---------+---------+-----+-------+-------+---------------+
only showing top 10 rows

+---------------+------+
|categoria_venta| count|
+---------------+------+
|           Alta|333617|
|         Normal|507530|
+--------

### Paso 7 Bonus: Media de ingresos por año
La columna `Date` viene en formato texto `M/d/yyyy`, por lo que se convierte a fecha con `to_date` y posteriormente se extrae el año con `year` para calcular el promedio anual de ingresos.

In [19]:
df_bonus_anual = df_ventas \
    .withColumn("fecha_convertida", to_date(trim(col("Date")), "M/d/yyyy")) \
    .withColumn("anio", year(col("fecha_convertida"))) \
    .groupBy("anio") \
    .agg(avg("Revenue").alias("media_ingresos")) \
    .orderBy("anio")
df_bonus_anual.show(20, truncate=False)

+----+------------------+
|anio|media_ingresos    |
+----+------------------+
|1999|420.5926656491961 |
|2000|401.734184770051  |
|2001|399.32866007253693|
|2002|417.9040199608311 |
|2003|423.9238212502331 |
|2004|437.4294183257592 |
|2005|457.5285688418277 |
|2006|500.5433832159752 |
|2007|514.123947315254  |
|2008|520.9093188860004 |
|2009|579.8277477172883 |
|2010|625.8745293617361 |
|2011|652.3791332537456 |
|2012|686.1609041076957 |
|2013|712.7470211873309 |
|2014|653.8721260040514 |
|2015|628.7663273961015 |
+----+------------------+
